# Análise de diárias e faturamento por unidade

#Pergunta 1: A diretoria quer entender o comportamento das diárias praticadas na rede e a distribuição do faturamento entre as unidades. O que os dados revelam?

#Importação dos dados

In [ ]:
import pandas as pd

df_reservas = pd.read_csv('reservas_clean.csv', sep=';')
df_tipos = pd.read_csv('tipos_clean.csv', sep=';')
df_unidades = pd.read_csv('unidades_clean.csv', sep=';')

#Separando reservas válidas (que geraram receitam)

In [ ]:
df_reservas_validas = df_reservas[df_reservas['status_reserva'].isin(['Confirmada','Concluída'])]
df_reservas_validas.head()

#Juntando reservas com o tipo de quarto 

In [ ]:
df_reservas_valor = df_reservas_validas.merge(df_tipos, on="id_tipo_quarto")
df_reservas_valor.head()

#Faturamento de cada reserva (valor da diária x nº de dias de hospedagem)

In [ ]:
df_reservas_valor['faturamento_reserva'] = df_reservas_valor['valor_diaria_base'] * df_reservas_valor['qtd_diarias']
df_reservas_valor.head()

#Incluindo nome da unidade na tabela de reservas_valor

In [ ]:
df_reservas_valor_unidades= df_reservas_valor.merge(df_unidades, on="id_unidade")
df_reservas_valor_unidades.head()

#Comportamento das diárias

In [ ]:
media = df_reservas_valor["valor_diaria_base"].mean()
mediana = df_reservas_valor["valor_diaria_base"].median()
variancia = df_reservas_valor["valor_diaria_base"].var()
desvio_padrao = df_reservas_valor["valor_diaria_base"].std()
coef_var = desvio_padrao / media 
distancia = (media - mediana) / mediana

print(f"Média: {media:.2f}")
print(f"Mediana: {mediana:.2f}")
print(f"Variância: {variancia:.2f}")
print(f"Desvio padrão: {desvio_padrao:.2f}")
print(f"Coeficiente de variação: {coef_var:.2%}")
print(f"Distância entre média e mediana: {distancia:.2%}")

#Faturamento por unidade NaraHoteis

In [ ]:
df_faturamento_unidades = df_reservas_valor_unidades.groupby('nome_unidade')['faturamento_reserva'].sum().reset_index()
df_faturamento_unidades['percentual'] = (df_faturamento_unidades['faturamento_reserva'] / df_faturamento_unidades['faturamento_reserva'].sum()) * 100
df_faturamento_unidades

In [ ]:
print(df_faturamento_unidades.sort_values(by="faturamento_reserva", ascending=False))

Conclusão

#Pergunta 1: A diretoria quer entender o comportamento das diárias praticadas na rede e a distribuição do faturamento entre as unidades. O que os
dados revelam? 

R:Os dados mostram que as diárias têm muita variação, com média em torno de R$ 560 e mediana de R$ 580, indicando preços de diárias bem diferentes entre os hotéis. O faturamento está distribuído de forma relativamente equilibrada entre as principais unidades: Paraty, Friburgo, Santa Teresa, Teresópolis e Petrópolis concentram cerca de metade da receita total, enquanto unidades como Centro e Nova Iguaçu Centro têm participação bem menor. 

# Painel Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axs = plt.subplots(2, 2, figsize=(14, 8))

# 1 - Histograma das diárias
sns.histplot(df_reservas_valor["valor_diaria_base"], bins=20, ax=axs[0,0], color="skyblue")
axs[0,0].set_title("Distribuição das diárias", fontsize=13, fontweight="bold")
axs[0,0].set_xlabel("Valor da diária (R$)")
axs[0,0].set_ylabel("")

# 2 - Boxplot das diárias
sns.boxplot(y=df_reservas_valor["valor_diaria_base"], ax=axs[0,1], color="lightgreen")
axs[0,1].set_title("Boxplot das diárias", fontsize=13, fontweight="bold")
axs[0,1].set_ylabel("Valor da diária (R$)")

# 3 - Faturamento por unidade (barras)
sns.barplot(x="nome_unidade", y="faturamento", data=df_faturamento_unidades, ax=axs[1,0], color="orange")
axs[1,0].set_title("Faturamento por unidade", fontsize=13, fontweight="bold")
axs[1,0].set_xlabel("")
axs[1,0].set_ylabel("Faturamento (R$)")
axs[1,0].tick_params(axis="x", rotation=90)

# 4 - Métricas resumo com plt.text()
media = df_reservas_valor["valor_diaria_base"].mean()
mediana = df_reservas_valor["valor_diaria_base"].median()
desvio = df_reservas_valor["valor_diaria_base"].std()
variancia = df_reservas_valor["valor_diaria_base"].var()
coef_var = desvio / media

axs[1,1].axis("off")
texto = (
    f"Média: R$ {media:.2f}\n"
    f"Mediana: R$ {mediana:.2f}\n"
    f"Desvio padrão: R$ {desvio:.2f}\n"
    f"Variância: {variancia:.2f}\n"
    f"Coeficiente de variação: {coef_var:.2%}"
)
axs[1,1].text(0.05, 0.95, texto,
              transform=axs[1,1].transAxes,
              fontsize=12, verticalalignment="top",
              fontfamily="monospace")

# Título geral
fig.suptitle("Diárias e Faturamento", fontsize=16, fontweight="bold", y=1.02)

plt.tight_layout()
plt.show()

Comentário:

Esse painel mostra como estão as diárias e o faturamento das unidades da rede Nara Hotéis.

O histograma ajuda a ver que os preços das diárias variam bastante.As diárias têm variação, com média em torno de R$ 560 e mediana de R$ 580, indicando preços bem diferentes entre os hotéis.

O boxplot confirma essa variação e mostra que existem alguns valores bem acima da média (outliers).

O gráfico de barras mostra o faturamento de cada unidade, deixando claro que as unidades que concentram mais faturamento são Paraty, Friburgo, Santa Teresa, Teresópolis e Petrópolis.

O quadro de métricas resume os números principais (média, mediana, desvio, variância e coeficiente de variação), facilitando a leitura do comportamento das diárias.

Com isso, dá pra concluir que as diárias têm muita diferença entre os hotéis e que o faturamento está mais concentrado em algumas unidades, como Paraty, Friburgo, Santa Teresa, Teresópolis e Petrópolis.

# Análise de unidades operando fora da capacidade 

#Pergunta 2: Existem unidades operando além de sua capacidade? Quais o impactos disso para o negócio?

#Importação dos dados

In [ ]:
import pandas as pd
df_reservas = pd.read_csv( 'reservas_clean.csv', sep=';')
df_unidades = pd.read_csv( 'unidades_clean.csv' , sep=';')

#Filtrando reservas válidas

In [ ]:
df_reservas_validas = df_reservas[df_reservas['status_reserva'].isin(['Confirmada','Concluída'])]
df_reservas_validas.head()

#Período das reservas

In [ ]:
data_inicio_reserva = pd.to_datetime(df_reservas_validas['data_checkin']).min()
data_fim_reserva = pd.to_datetime(df_reservas_validas['data_checkout']).max()
periodo = (data_fim_reserva - data_inicio_reserva).days
print("Data início:", data_inicio_reserva)
print("Data fim:", data_fim_reserva)
print("Período em dias:", periodo)

#Contando o nº de reservas por dia

In [ ]:
df_reservas_por_dia = df_reservas_validas.groupby(['id_unidade', 'data_checkin']).size().reset_index()
df_reservas_por_dia.columns = ['id_unidade', 'data_checkin', 'reservas_no_dia']
df_reservas_por_dia.head()

#Capacidade dos hotéis: o número de quartos fazendo o merge com a tabela de unidades

In [ ]:
df_capacidade_unidades = df_reservas_por_dia.merge(df_unidades[['id_unidade','nome_unidade','num_quartos_total']], on='id_unidade')
df_capacidade_unidades.head()

#Verificando se o nº de reservas/check ins no dia é superior ao nº de quartos da unidade para identificar se há overbooking.

In [ ]:
df_unidades_com_overbooking = df_capacidade_unidades[df_capacidade_unidades['reservas_no_dia'] > df_capacidade_unidades['num_quartos_total']]
df_unidades_com_overbooking.head()

Conclusão

#Pergunta 2: Existem unidades operando além de sua capacidade? Quais os impactos disso para o negócio?

R:Sim, existem unidades operando além da capacidade. Encontramos 4 datas em que o número de pessoas chegando para entrar no hotel foi maior do que a quantidade de quartos que o hotel possui: o NaraHoteis Centro estourou a capacidade duas vezes (em 15/07/2023 com 74 reservas e em 15/01/2024 com 79 reservas, sendo que ele só tem 60 quartos).O NaraHoteis Nova Iguaçu Centro também estourou duas vezes (em 15/10/2023 com 62 reservas e em 15/06/2024 com 64 reservas, para um total de 50 quartos).
Impactos para o negócio são overbooking real, reembolso do valor pago pela reserva que não pode ser honrada,clientes insatisfeitos,além de sobrecarga da equipe do hotel.

# Análise de performance de receita por unidade

#Pergunta 3: Existem unidades com performance de receita discrepante em relação às demais? Como você identificaria, mediria e comprovaria isso?

#Importação dos dados

In [ ]:
import pandas as pd
import numpy as np

df_reservas = pd.read_csv("reservas_clean.csv", sep=";")
df_unidades = pd.read_csv("unidades_clean.csv", sep=";")
df_quartos = pd.read_csv("tipos_clean.csv", sep=";")

#Filtrando as reservas válidas

In [ ]:
df_reservas_validas = df_reservas[df_reservas['status_reserva'].isin(['Confirmada','Concluída'])]
df_reservas_validas.head()

#Valor da diária pelo tipo do quarto 

In [ ]:
df_reservas_validas = df_reservas_validas.merge(df_quartos[['id_tipo_quarto','valor_diaria_base','descricao']],on='id_tipo_quarto')
df_reservas_validas.head()

#Receita de cada reserva


In [ ]:
df_reservas_validas['receita'] = df_reservas_validas['qtd_diarias'] * df_reservas_validas['valor_diaria_base']
df_reservas_validas.head()

#Total de receita por unidade

In [ ]:
df_receita_unidade = df_reservas_validas.groupby('id_unidade')['receita'].sum().reset_index()
df_receita_unidade

#Total de receita por Unidade

In [ ]:
df_receita_unidade = df_receita_unidade.merge(df_unidades[['id_unidade','nome_unidade',]], on='id_unidade')
df_receita_unidade.head()

#Verificando Outliers

In [ ]:
array_df_receita_unidade=np.array(df_receita_unidade['receita'])

Q1 = np.percentile(array_df_receita_unidade, 25)
Q3 = np.percentile(array_df_receita_unidade, 75)
IQR = Q3 - Q1
limite_inferior = Q1 - (1.5 * IQR)
limite_superior = Q3 + (1.5 * IQR)
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Limite inferior:", limite_inferior)
print("Limite superior:", limite_superior)

#Demarcando outliers

In [ ]:
df_receita_unidade['outlier'] = (df_receita_unidade['receita'] < limite_inferior) | (df_receita_unidade['receita'] > limite_superior)
df_receita_unidade

In [ ]:
print(df_receita_unidade)

Conclusão

#Pergunta 3: Existem unidades com performance de receita discrepante em relação às demais? Como você identificaria, mediria e comprovaria isso?

Sim. As unidades NaraHoteis Centro e NaraHoteis Nova Iguaçu Centro tem receita discrepante e inferior em relação às demais.Identificamos isso calculando o IQR e destacando os outliers.

# Análise entre RevPAR das unidades e a avaliação média dos hóspedes

#Pergunta 4: Há relação entre o RevPAR das unidades e a avaliação média dos hóspedes? Essa relação é forte o suficiente para sustentar uma análise
preditiva?

#Importação dos dados

In [ ]:
import pandas as pd
import numpy as np
df_reservas = pd.read_csv("reservas_clean.csv", sep=";")
df_unidades = pd.read_csv("unidades_clean.csv", sep=";")
df_quartos = pd.read_csv("tipos_clean.csv", sep=";")

#Filtrando apenas as reservas válidas

In [ ]:
df_reservas_validas = df_reservas[df_reservas['status_reserva'].isin(['Confirmada','Concluída'])]
df_reservas_validas.head()

#Valor da diária das reservas válidas

In [ ]:
df_reservas_validas = df_reservas_validas.merge(df_quartos[['id_tipo_quarto','valor_diaria_base','descricao']], on='id_tipo_quarto')
df_reservas_validas.head()

#Corrigindo o valor_diaria_base para número

In [ ]:
df_reservas_validas['valor_diaria_base'] = df_reservas_validas['valor_diaria_base'].astype(str)
df_reservas_validas['valor_diaria_base'] = df_reservas_validas['valor_diaria_base'].str.replace('R$','')
df_reservas_validas['valor_diaria_base'] = df_reservas_validas['valor_diaria_base'].str.replace('.','')
df_reservas_validas['valor_diaria_base'] = df_reservas_validas['valor_diaria_base'].str.replace(',','.')
df_reservas_validas['valor_diaria_base'] = df_reservas_validas['valor_diaria_base'].astype(float)
df_reservas_validas['qtd_diarias'] = pd.to_numeric(df_reservas_validas['qtd_diarias'], errors='coerce')
df_reservas_validas.head()

#Valor total de cada reserva

In [ ]:
df_reservas_validas['valor_total_reserva'] = df_reservas_validas['qtd_diarias'] * df_reservas_validas['valor_diaria_base']
df_reservas_validas.head()

#Juntando reservas com unidades para trazer o nome_unidade

In [ ]:
df_reservas = df_reservas_validas.merge(df_unidades[['id_unidade','nome_unidade']], on='id_unidade')
df_reservas.head()

#Receita de cada unidade

In [ ]:
df_receita_unidade = df_reservas.groupby('nome_unidade')['valor_total_reserva'].sum().reset_index()
df_receita_unidade = df_receita_unidade.rename(columns={'valor_total_reserva': 'receita_total'})
df_receita_unidade.head()

#Avaliação média por unidade

In [ ]:
df_avaliacao_unidade = df_reservas.groupby('nome_unidade')['avaliacao_hospede'].mean().reset_index(name="avaliacao_hospede_media")
df_avaliacao_unidade

#Lado a lado: receita de cada unidade vs avaliação média dos hóspedes

In [ ]:
df_unidade = df_receita_unidade.merge(df_avaliacao_unidade, on='nome_unidade')
df_unidade

#Juntando com o número de quartos

In [ ]:
df_unidade = df_unidade.merge(df_unidades[['num_quartos_total', 'nome_unidade']], on='nome_unidade')
df_unidade.head()

#RevPAR

In [ ]:
periodo=365
df_unidade['RevPAR'] = df_unidade['receita_total'] / (df_unidade['num_quartos_total'] * periodo)
df_unidade.head()


#Covariância entre RevPAR e avaliação média dos hóspedes

In [ ]:
matriz_cov_revpar_avaliacao= np.cov(df_unidade['RevPAR'], df_unidade['avaliacao_hospede_media'])
print(matriz_cov_revpar_avaliacao)

# Extraindo só o valor da covariância
cov_valor = matriz_cov_revpar_avaliacao[0, 1]
print(f"Covariância: {cov_valor:.2f}")

#Analisando a correlação entre RevPAR e a avaliação média dos hóspedes

In [ ]:
correlacao_revpar_avaliacao_media_hospedes = np.corrcoef(df_unidade['RevPAR'], df_unidade['avaliacao_hospede_media'])
print(f"Pearson r = {correlacao_revpar_avaliacao_media_hospedes[0,1]:.4f}")

Conclusão

#Pergunta 4: Há relação entre o RevPAR das unidades e a avaliação média dos hóspedes? Essa relação é forte o suficiente para sustentar uma análise preditiva?

R: Há correlação moderada positiva entre RevPAR e avaliação média dos hóspedes (correlação = 0,68). A correlação permite sustentar uma análise preditiva.

# Análise preditiva RevPAR a partir da avaliação média dos hóspedes

#Pergunta 5: Com base nos dados históricos, é possível estimar o RevPAR esperado de uma unidade a partir da sua avaliação média? Construa e avalie esse modelo. 

#Importando dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

x=df_unidade[['avaliacao_hospede_media']]
y=df_unidade['RevPAR']

#Modelo preditivo

In [ ]:
modelo = LinearRegression()
modelo.fit(x, y)

In [ ]:
x_extendido = pd.DataFrame({'avaliacao_hospede_media':[9, 9.2,9.5,10]})
y_tend = modelo.predict(x_extendido)

#Visualização de resultados

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(
df_unidade['avaliacao_hospede_media'],
df_unidade['RevPAR'],
color='green',
label='Dados reais'
)
plt.plot(
x_extendido['avaliacao_hospede_media'],
y_tend,
color='orange',
label='Reta de tendência'
)

Conclusão

#Pergunta 5: Com base nos dados históricos, é possível estimar o RevPAR esperado de uma unidade a partir da sua avaliação média? Construa e avalie esse modelo.

R:Sim, é possível. Foi construído um modelo de regressão linear relacionando a avaliação média dos hóspedes com o RevPAR das unidades. O gráfico mostra uma tendência positiva: quanto maior a avaliação, maior tende a ser o RevPAR. Isso indica que a satisfação dos hóspedes tem impacto na RevPAR.

# Análise de Variabilidade de Faturamento por Região

#Pergunta 6: Qual região apresenta maior variabilidade no faturamento? O que isso pode indicar para a gestão comercial?

#Exibindo a regiao das unidades e o total de receita de cada uma.

In [ ]:
df_receita_regiao = df_receita_unidade.merge(df_unidades[['id_unidade','regiao']], on='id_unidade')
df_receita_regiao.head()

#Analisando variabilidade por região

In [ ]:
#Capital
capital = df_receita_regiao[df_receita_regiao["regiao"] == "Capital"]["receita"]
media_capital = np.mean(capital)
desvio_capital = np.std(capital)
print("Região: Capital")
print(f"Média da receita: {media_capital:.2f}")
print(f"Desvio padrão: {desvio_capital:.2f}")
print(f"Faixa típica: {media_capital-desvio_capital:.2f} até {media_capital+desvio_capital:.2f}\n")

#Baixada Fluminense
baixada = df_receita_regiao[df_receita_regiao["regiao"] == "Baixada Fluminense"]["receita"]
media_baixada = np.mean(baixada)
desvio_baixada = np.std(baixada)
print("Região: Baixada Fluminense")
print(f"Média da receita: {media_baixada:.2f}")
print(f"Desvio padrão: {desvio_baixada:.2f}")
print(f"Faixa típica: {media_baixada-desvio_baixada:.2f} até {media_baixada+desvio_baixada:.2f}\n")

#Serra
serra = df_receita_regiao[df_receita_regiao["regiao"] == "Serra"]["receita"]
media_serra = np.mean(serra)
desvio_serra = np.std(serra)
print("Região: Serra")
print(f"Média da receita: {media_serra:.2f}")
print(f"Desvio padrão: {desvio_serra:.2f}")
print(f"Faixa típica: {media_serra-desvio_serra:.2f} até {media_serra+desvio_serra:.2f}\n")

#Costa Verde
costa = df_receita_regiao[df_receita_regiao["regiao"] == "Costa Verde"]["receita"]
media_costa = np.mean(costa)
desvio_costa = np.std(costa)
print("Região: Costa Verde")
print(f"Média da receita: {media_costa:.2f}")
print(f"Desvio padrão: {desvio_costa:.2f}")
print(f"Faixa típica: {media_costa-desvio_costa:.2f} até {media_costa+desvio_costa:.2f}\n")

Conclusão

#Pergunta 6: Qual região apresenta maior variabilidade no faturamento? O que isso pode indicar para a gestão comercial?

A região com maior variabilidade no faturamento é a Baixada Fluminense, seguida pela Capital. Isso acontece porque as unidades dessas regiões têm receitas bem diferentes entre si. Isso indica que essas regiões são mais desiguais e precisam de mais atenção para entender o motivo dessas diferenças.

# Painel Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(2, 2, figsize=(14, 8))

# Variabilidade por região
df_stats_regiao = df_receita_regiao.groupby("regiao")["receita"].agg(["mean","std"]).reset_index()

sns.barplot(
    x="regiao", y="mean",
    data=df_stats_regiao,
    ax=axs[0,0],
    color="skyblue"
)

axs[0,0].set_title("Variabilidade por região")
axs[0,0].tick_params(axis="x", rotation=45)

# Receita por região
sns.barplot(
    x="regiao", y="receita",
    data=df_receita_regiao,
    ax=axs[0,1],
    color="lightgreen",
    estimator=sum
)

axs[0,1].set_title("Receita por região")
axs[0,1].tick_params(axis="x", rotation=45)

# Receita por unidade
df_ord = df_receita_unidade.sort_values("receita", ascending=False)

sns.barplot(
    x="nome_unidade", y="receita",
    data=df_ord,
    ax=axs[1,0],
    color="orange"
)

axs[1,0].set_title("Receita por unidade")
axs[1,0].tick_params(axis="x", rotation=90)

# Texto (outliers)
axs[1,1].axis("off")

outliers = df_receita_unidade[
    df_receita_unidade["outlier"] == True
]["nome_unidade"].tolist()

texto = "Outliers:\n" + "\n".join(outliers)

axs[1,1].text(
    0.05, 0.95,
    texto,
    transform=axs[1,1].transAxes,
    fontsize=12,
    verticalalignment='top'
)

# Título geral
fig.suptitle("Painel de Receita", fontsize=16, fontweight="bold")

plt.tight_layout()
plt.show()




Comentário:

Gráfico 1 — Variabilidade da receita por região:
Esse gráfico mostra que algumas regiões têm muita variação na receita. A região com maior variabilidade no faturamento é a Baixada Fluminense, seguida pela Capital, pois as unidades têm receitas muito diferentes entre si.

Gráfico 2 — Receita total por região:
Neste gráfico é possível identificar as regiões com maior faturamento (Costa Verde,Serra e Capital) e que são importantes para o desempenho da empresa.

Gráfico 3 — Receita por unidade:
Esse gráfico mostra quanto cada unidade gera de faturamento. Dá pra perceber que algumas unidades têm um faturamento bem menor que outras (Centro e Nova Iguaçú Centro), o que mostra diferença de desempenho entre elas.

Gráfico 4 — Outliers:
Nesse quadro destacamos as unidades que estão fora do padrão. Ou seja, unidades que têm receita muito diferente das outras.As unidades NaraHoteis Centro e NaraHoteis Nova Iguaçu Centro tem receita discrepante e inferior em relação às demais.

Conclusão:
No geral, o painel ajuda a entender como está a receita da empresa, mostrando diferenças entre regiões e unidades. Isso ajuda a identificar onde estão os melhores resultados e pontos de melhoria.